# FPL Season-to-Season Valuation
Matches returning players by name only, so players reclassified to a different
position between seasons (e.g., MID to FWD) are still included. The output table uses 25/26 for clarity.

**Parameters to set below:** paths to the two CSVs and the ±% band for undervalued/overvalued labels.


In [1]:
# Parameters
prev_csv = '24-25.csv'
curr_csv = '25-26.csv'
pct_band = 0.05


In [2]:
# Imports
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import KFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import ElasticNet
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score




In [3]:
# Utilities
def make_key_no_position(df: pd.DataFrame) -> pd.Series:
    for col in ['first_name','second_name']:
        if col not in df.columns:
            df[col] = ''
    return (
        df['first_name'].astype(str).str.strip().str.lower() + '|' +
        df['second_name'].astype(str).str.strip().str.lower()
    )


def compute_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred, squared=False)
    r2 = r2_score(y_true, y_pred)
    mape = (np.abs(y_true - y_pred) / np.maximum(1e-8, y_true)).mean()
    return {'MAE': mae, 'RMSE': rmse, 'R2': r2, 'MAPE': mape}


def label_from_band(p, band):
    if p > band:
        return 'Undervalued (actual below predicted)'
    elif p < -band:
        return 'Overvalued (actual above predicted)'
    else:
        return 'Appropriately valued'


##  Load and match by name only

In [4]:
df_prev = pd.read_csv(prev_csv)
df_curr = pd.read_csv(curr_csv)

df_prev['join_key'] = make_key_no_position(df_prev)
df_curr['join_key'] = make_key_no_position(df_curr)

overlap = set(df_prev['join_key']).intersection(set(df_curr['join_key']))
prev = df_prev[df_prev['join_key'].isin(overlap)].copy()
curr = df_curr[df_curr['join_key'].isin(overlap)].copy()

print('Prev rows:', len(df_prev), 'Curr rows:', len(df_curr))
print('Overlap rows:', len(prev))

# Filter: require prior-season history
if 'total_points' in prev.columns:
    prev = prev[prev['total_points'].fillna(0) > 0].copy()
print('After 24/25 history filter:', len(prev))


Prev rows: 804 Curr rows: 668
Overlap rows: 456
After 24/25 history filter: 352


##  Merge seasons and select features

In [5]:
merged = prev.merge(
    curr[['join_key','now_cost','first_name','second_name','element_type']],
    on='join_key', how='inner', suffixes=('_2425','_2526')
).rename(columns={'now_cost':'now_cost_2526', 'element_type':'element_type_2526'})

base_feats = [
    'total_points','minutes','goals_scored','assists',
    'goals_conceded','clean_sheets','ict_index','influence',
    'creativity','threat','bonus','bps','red_cards','yellow_cards',
    'selected_by_percent'
]
feature_cols = []
if 'now_cost_2425' in merged.columns:
    feature_cols.append('now_cost_2425')
feature_cols.extend([c for c in base_feats if c in merged.columns])

merged = merged.dropna(subset=feature_cols + ['now_cost_2526']).copy()
print('Training rows:', len(merged))
print('Features:', feature_cols)


Training rows: 352
Features: ['now_cost_2425', 'total_points', 'minutes', 'goals_scored', 'assists', 'goals_conceded', 'clean_sheets', 'ict_index', 'influence', 'creativity', 'threat', 'bonus', 'bps', 'red_cards', 'yellow_cards', 'selected_by_percent']


##  Train models (5-fold CV) and compare

In [6]:
X = merged[feature_cols].copy()
y = merged['now_cost_2526'].astype(float).copy()

enet = Pipeline([('scaler', StandardScaler()),
                 ('model', ElasticNet(alpha=0.01, l1_ratio=0.2, random_state=42, max_iter=10000))])
rf   = Pipeline([('model', RandomForestRegressor(n_estimators=400, max_depth=None,
                                                 min_samples_leaf=2, random_state=42, n_jobs=-1))])

kf = KFold(n_splits=5, shuffle=True, random_state=42)
pred_en = cross_val_predict(enet, X, y, cv=kf)
pred_rf = cross_val_predict(rf,   X, y, cv=kf)

m_en = compute_metrics(y, pred_en)
m_rf = compute_metrics(y, pred_rf)

pd.DataFrame([
    {'Model':'ElasticNet', **m_en},
    {'Model':'RandomForest', **m_rf}
])


,Model,MAE,RMSE,R2,MAPE
0,ElasticNet,2.573354,3.410503,0.927032,0.047570
1,RandomForest,2.824045,4.483355,0.873904,0.049868


## Fit best model, predict, label, and build results

In [7]:
best_name, best = ('RandomForest', rf) if m_rf['MAE'] <= m_en['MAE'] else ('ElasticNet', enet)
best.fit(X, y)

merged['predicted_now_cost_2526'] = best.predict(X)
merged['pct_diff'] = (merged['predicted_now_cost_2526'] - merged['now_cost_2526']) / merged['now_cost_2526']
merged['valuation'] = merged['pct_diff'].apply(lambda p: label_from_band(p, pct_band))

merged['player_name'] = (merged['first_name_2526'].astype(str) + ' ' + merged['second_name_2526'].astype(str)).str.strip()

cols_out = ['player_name','element_type_2526'] + feature_cols + [
    'now_cost_2526','predicted_now_cost_2526','pct_diff','valuation'
]
results = merged[cols_out].sort_values('pct_diff', ascending=False).reset_index(drop=True)

print('Best model:', best_name)
results.head(10)


Best model: ElasticNet


,player_name,element_type_2526,now_cost_2425,total_points,minutes,goals_scored,assists,goals_conceded,clean_sheets,ict_index,...,threat,bonus,bps,red_cards,yellow_cards,selected_by_percent,now_cost_2526,predicted_now_cost_2526,pct_diff,valuation
0,Nicolas Jackson,FWD,77,121,2220,10,6,24,11,187.4,...,976.0,17,417,1,7,6.2,65,79.343092,0.220663,Undervalued (actual below predicted)
1,Marc Guéhi,DEF,47,116,3059,3,2,42,11,131.0,...,247.0,11,492,1,7,5.7,45,52.035744,0.156350,Undervalued (actual below predicted)
2,Rodrigo Martins Gomes,DEF,51,42,794,2,0,20,1,47.6,...,113.0,0,183,0,0,0.1,45,51.993265,0.155406,Undervalued (actual below predicted)
3,Bryan Mbeumo,MID,83,236,3415,20,9,55,9,340.8,...,1060.0,29,948,0,3,46.7,80,92.026694,0.150334,Undervalued (actual below predicted)
4,Eiran Cashin,DEF,45,1,19,0,0,2,0,0.7,...,0.0,0,-2,0,0,0.0,40,45.963743,0.149094,Undervalued (actual below predicted)
5,Andrés García,DEF,45,8,316,0,0,5,0,12.0,...,7.0,0,26,0,0,0.1,40,45.776472,0.144412,Undervalued (actual below predicted)
6,Diogo Dalot Teixeira,DEF,50,95,2812,0,4,45,8,118.2,...,258.0,9,352,0,5,4.7,45,51.474850,0.143886,Undervalued (actual below predicted)
7,Tyrell Malacia,DEF,44,2,99,0,0,1,0,1.7,...,0.0,0,2,0,1,0.0,40,45.265797,0.131645,Undervalued (actual below predicted)
8,Mykhailo Mudryk,MID,61,7,145,0,0,2,0,8.5,...,31.0,0,26,0,0,0.2,50,56.030726,0.120615,Undervalued (actual below predicted)
9,Álex Moreno Lopera,DEF,42,34,952,0,1,16,3,36.4,...,29.0,1,126,0,2,0.4,40,44.778788,0.119470,Undervalued (actual below predicted)


## Export CSVs and plots

In [8]:
all_path   = os.path.join( 'all_player_valuations_25_26.csv')
under_path = os.path.join( 'top_undervalued_25_26.csv')
over_path  = os.path.join('top_overvalued_25_26.csv')

results.to_csv(all_path, index=False)
results[results['valuation'].str.startswith('Undervalued')].head(15).to_csv(under_path, index=False)
results[results['valuation'].str.startswith('Overvalued')].head(15).to_csv(over_path, index=False)

plt.figure()
plt.scatter(results['now_cost_2526'], results['predicted_now_cost_2526'])
max_val = max(results['now_cost_2526'].max(), results['predicted_now_cost_2526'].max())
plt.plot([0, max_val], [0, max_val])
plt.xlabel('Actual 25/26 now_cost')
plt.ylabel('Predicted 25/26 now_cost')
plt.title(f'Actual vs Predicted ({best_name})')
scatter_path = os.path.join( f'actual_vs_predicted_{best_name.lower()}_25_26.png')
plt.savefig(scatter_path, bbox_inches='tight')
plt.close()

print('Saved:')
print(all_path)
print(under_path)
print(over_path)
print(scatter_path)


Saved:
all_player_valuations_25_26.csv
top_undervalued_25_26.csv
top_overvalued_25_26.csv
actual_vs_predicted_elasticnet_25_26.png


## 6) Explainability (ElasticNet coefficients or RF importances)

In [9]:
if best_name == 'RandomForest':
    fi = pd.Series(best.named_steps['model'].feature_importances_, index=feature_cols).sort_values(ascending=False)
    fi.head(12)

    plt.figure()
    fi.head(12).plot(kind='bar')
    plt.title('Top Feature Importances (RandomForest)')
    plt.ylabel('Relative Importance')
    feat_imp_path = os.path.join( 'feature_importances_rf.png')
    plt.tight_layout()
    plt.savefig(feat_imp_path, bbox_inches='tight')
    plt.close()
    print('Saved:', feat_imp_path)
else:
    coef = best.named_steps['model'].coef_
    coef_df = pd.DataFrame({'feature': feature_cols, 'coef': coef, 'abs_coef': np.abs(coef)}).sort_values('abs_coef', ascending=False)
    coef_df.head(20)

    coef_path = os.path.join( 'elasticnet_coefficients_25_26.csv')
    coef_df.to_csv(coef_path, index=False)

    plt.figure()
    coef_df.head(12).set_index('feature')['abs_coef'].plot(kind='bar')
    plt.title('ElasticNet | Top Coefficients by Magnitude (std-scaled)')
    plt.ylabel('|Coefficient|')
    coef_plot_path = os.path.join( 'elasticnet_top_coefs.png')
    plt.tight_layout()
    plt.savefig(coef_plot_path, bbox_inches='tight')
    plt.close()
    print('Saved:', coef_path)
    print('Saved:', coef_plot_path)


Saved: elasticnet_coefficients_25_26.csv
Saved: elasticnet_top_coefs.png


##  Quick check for specific players (e.g., Jarrod Bowen)

In [10]:
results[results['player_name'].str.lower().str.contains('bowen')].head(3)

,player_name,element_type_2526,now_cost_2425,total_points,minutes,goals_scored,assists,goals_conceded,clean_sheets,ict_index,...,threat,bonus,bps,red_cards,yellow_cards,selected_by_percent,now_cost_2526,predicted_now_cost_2526,pct_diff,valuation
65,Jarrod Bowen,FWD,79,193,2974,13,11,50,8,271.9,...,1081.0,21,678,0,1,18.0,80,84.621582,0.05777,Undervalued (actual below predicted)
